[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/diogoflim/Pesquisa-Operacional-III-A/blob/main/14_Intro_SimPy.ipynb)

## **Pesquisa Operacional III-A**

**Professor:**
- Diogo Ferreira de Lima Silva (TEP-UFF)

# Processos com Múltiplas Atividades no SimPy

---

## Onde estamos

| Aula | Conteúdo |
|------|----------|
| 12 | Funções geradoras, SimPy básico (Pomodoro, recurso simples) |
| 13 | Filas M/M/s, monitoramento de recursos |
| **14** | **Processos com múltiplas atividades, coleta de estatísticas e replicações** |

Nos notebooks 12 e 13 o foco era em sistemas com **uma única atividade de serviço**. Na prática, processos reais encadeiam várias etapas, cada uma com seu próprio recurso e distribuição de tempo.

Neste notebook vamos:
1. Modelar um processo com **múltiplas atividades em sequência**.
2. Coletar estatísticas **por etapa** (espera e processamento em cada serviço).
3. Analisar **roteamento condicional** (fluxo que se divide conforme uma probabilidade).
4. Executar **múltiplas replicações** e calcular intervalos de confiança.

In [ ]:
# !pip install simpy   # descomente se necessário

import simpy
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

---

# Parte 1 — Processo com Múltiplas Atividades

## Exemplo: Salão de Beleza

Um salão oferece três serviços em sequência:

| Etapa | Recurso | Tempo de serviço |
|-------|---------|------------------|
| A — Recepção e diagnóstico | 1 colaborador | constante: 3 min |
| B — Tratamento principal   | 1 colaborador | constante: 8 min |
| C — Finalização            | 1 colaborador | exponencial média 4 min |

**Chegadas:** processo de Poisson com $\lambda = 0{,}2$ clientes/min (intervalo médio = 5 min).

Cada cliente precisa dos três serviços nessa ordem. Se o próximo colaborador estiver ocupado, o cliente espera em fila.

### Modelando o processo

Vamos construir o modelo em três etapas:
1. Criar o processo do cliente.
2. Criar o processo de chegadas.
3. Executar e coletar estatísticas.

In [ ]:
def salao(env, nome, colab_A, colab_B, colab_C, stats):
    """Processo de um cliente no salão de beleza."""
    chegada_total = env.now

    # --- Etapa A ---
    t_ini_fila_A = env.now
    with colab_A.request() as req:
        yield req
        stats["espera_A"].append(env.now - t_ini_fila_A)
        t_ini_proc = env.now
        yield env.timeout(3)
        stats["proc_A"].append(env.now - t_ini_proc)

    # --- Etapa B ---
    t_ini_fila_B = env.now
    with colab_B.request() as req:
        yield req
        stats["espera_B"].append(env.now - t_ini_fila_B)
        t_ini_proc = env.now
        yield env.timeout(8)
        stats["proc_B"].append(env.now - t_ini_proc)

    # --- Etapa C ---
    t_ini_fila_C = env.now
    with colab_C.request() as req:
        yield req
        stats["espera_C"].append(env.now - t_ini_fila_C)
        t_ini_proc = env.now
        yield env.timeout(np.random.exponential(4))
        stats["proc_C"].append(env.now - t_ini_proc)

    stats["tempo_total"].append(env.now - chegada_total)


def chegadas_salao(env, colab_A, colab_B, colab_C, stats):
    i = 1
    while True:
        yield env.timeout(np.random.exponential(5))
        env.process(salao(env, f"Cliente {i}", colab_A, colab_B, colab_C, stats))
        i += 1

In [ ]:
def nova_simulacao_salao(seed=42, T=6000):
    """Executa uma replicação e retorna o dicionário de estatísticas."""
    np.random.seed(seed)
    stats = {k: [] for k in ["espera_A", "espera_B", "espera_C",
                               "proc_A", "proc_B", "proc_C", "tempo_total"]}
    env = simpy.Environment()
    A = simpy.Resource(env, capacity=1)
    B = simpy.Resource(env, capacity=1)
    C = simpy.Resource(env, capacity=1)
    env.process(chegadas_salao(env, A, B, C, stats))
    env.run(until=T)
    return stats

stats = nova_simulacao_salao(seed=42, T=6000)

resumo = pd.DataFrame({
    "Etapa" : ["A", "B", "C"],
    "Espera média (min)"       : [np.mean(stats["espera_A"]), np.mean(stats["espera_B"]), np.mean(stats["espera_C"])],
    "Processamento médio (min)": [np.mean(stats["proc_A"]),   np.mean(stats["proc_B"]),   np.mean(stats["proc_C"])],
}).round(2)

print(resumo.to_string(index=False))
print(f"\nClientes que concluíram o processo : {len(stats['tempo_total'])}")
print(f"Tempo médio total no salão         : {np.mean(stats['tempo_total']):.2f} min")

In [ ]:
# Gráfico de espera por etapa
etapas  = ["Etapa A", "Etapa B", "Etapa C"]
esperas = [np.mean(stats["espera_A"]), np.mean(stats["espera_B"]), np.mean(stats["espera_C"])]
procs   = [np.mean(stats["proc_A"]),   np.mean(stats["proc_B"]),   np.mean(stats["proc_C"])]

x = np.arange(len(etapas))
fig, ax = plt.subplots(figsize=(8, 4))
bars1 = ax.bar(x - 0.2, esperas, 0.4, label="Espera em fila", color="orange", edgecolor="black")
bars2 = ax.bar(x + 0.2, procs,   0.4, label="Processamento", color="steelblue", edgecolor="black")
ax.set_xticks(x)
ax.set_xticklabels(etapas)
ax.set_ylabel("Minutos")
ax.set_title("Tempo médio de espera e processamento por etapa")
ax.legend()
ax.grid(True, axis="y")
plt.tight_layout()
plt.show()

**Interpretação:** qual etapa é o **gargalo** do processo? Qual tem a maior espera? Isso é consistente com o tempo de processamento mais longo?

**Experimento:** altere a capacidade dos recursos (mais colaboradores em B ou C) e observe o impacto nas esperas.

---

# Parte 2 — Roteamento Condicional

Em muitos processos reais, nem todo cliente segue o mesmo caminho. Vamos modelar um processo em que **40% dos clientes** passam por uma etapa adicional (B2 — tratamento especial, 12 min) antes de continuar.

```
             ┌──── B2 (40%) ────┐
Chegada → A ─┤                  ├─→ C → Saída
             └───── (60%) ──────┘
```

Todos os colaboradores compartilham o mesmo recurso R1 exceto B2, que tem o seu próprio (R2).

In [ ]:
import random

def processo_com_roteamento(env, nome, R1, R2, stats_rot):
    """Cliente passa por A → (B2 com prob. 0.4) → C."""
    chegada = env.now

    # Etapa A (R1, 10 min)
    with R1.request() as req:
        yield req
        yield env.timeout(10)

    # Etapa B2 — opcional (R1, 12 min)
    passou_por_B2 = random.random() < 0.4
    if passou_por_B2:
        with R1.request() as req:
            yield req
            yield env.timeout(12)

    # Etapa C (R1, 8 min)
    with R1.request() as req:
        yield req
        yield env.timeout(8)

    stats_rot["tempo_total"].append(env.now - chegada)
    stats_rot["passou_B2"].append(int(passou_por_B2))


def chegadas_roteamento(env, R1, R2, stats_rot):
    i = 1
    while True:
        yield env.timeout(50)    # chegada a cada 50 min (processo espaçado)
        env.process(processo_com_roteamento(env, f"Trabalho {i}", R1, R2, stats_rot))
        i += 1


random.seed(10)
np.random.seed(10)

stats_rot = {"tempo_total": [], "passou_B2": []}
env_rot   = simpy.Environment()
R1        = simpy.Resource(env_rot, capacity=1)
R2        = simpy.Resource(env_rot, capacity=1)
env_rot.process(chegadas_roteamento(env_rot, R1, R2, stats_rot))
env_rot.run(until=100_000)

tc_com_b2 = [tc for tc, b2 in zip(stats_rot["tempo_total"], stats_rot["passou_B2"]) if b2]
tc_sem_b2 = [tc for tc, b2 in zip(stats_rot["tempo_total"], stats_rot["passou_B2"]) if not b2]

print(f"Trabalhos concluídos            : {len(stats_rot['tempo_total'])}")
print(f"Passaram por B2 (~40%)          : {sum(stats_rot['passou_B2'])}")
print(f"TC médio (com B2)               : {np.mean(tc_com_b2):.2f} min")
print(f"TC médio (sem B2)               : {np.mean(tc_sem_b2):.2f} min")
print(f"TC médio geral                  : {np.mean(stats_rot['tempo_total']):.2f} min")

### Calculando o TC esperado analiticamente

Com chegadas a cada 50 min (sem fila), o tempo de ciclo esperado é:

$$\text{TC esperado} = 10 + 0{,}4 \times 12 + 8 = 22{,}8 \text{ min}$$

O resultado simulado deve ficar próximo desse valor (a pequena diferença se deve ao compartilhamento de R1).

---

# Parte 3 — Múltiplas Replicações e Variabilidade

Uma única simulação é um **experimento aleatório**: resultados diferentes surgem a cada execução. Para obter estimativas confiáveis, precisamos de **múltiplas replicações** com sementes distintas.

**Por que não usar uma simulação muito longa?**

- A simulação longa estima a média no **estado estacionário**, mas não captura a **variabilidade natural** do sistema.
- Replicações independentes permitem **calcular intervalos de confiança** para as métricas de interesse.

In [ ]:
N_REP   = 100
T_SIM   = 6_000   # minutos por replicação

medias_totais = []
medias_C      = []

for rep in range(N_REP):
    s = nova_simulacao_salao(seed=rep, T=T_SIM)
    if s["tempo_total"]:   # ignora replicações sem dados
        medias_totais.append(np.mean(s["tempo_total"]))
        medias_C.append(np.mean(s["espera_C"]))

print(f"Replicações com dados: {len(medias_totais)}")
print()
print("=== Tempo total no salão ===")
print(f"  Média das replicações    : {np.mean(medias_totais):.2f} min")
print(f"  Desvio padrão            : {np.std(medias_totais):.2f} min")
z = 1.96
ep = z * np.std(medias_totais) / np.sqrt(len(medias_totais))
print(f"  IC 95%                   : [{np.mean(medias_totais) - ep:.2f}, {np.mean(medias_totais) + ep:.2f}]")
print()
print("=== Espera na etapa C ===")
print(f"  Média das replicações    : {np.mean(medias_C):.2f} min")
print(f"  Desvio padrão            : {np.std(medias_C):.2f} min")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].hist(medias_totais, bins=20, edgecolor='black', color='steelblue')
axes[0].axvline(np.mean(medias_totais), color='red', linestyle='--', label=f"Média = {np.mean(medias_totais):.1f}")
axes[0].set_title("Distribuição do tempo médio no salão (100 replicações)")
axes[0].set_xlabel("Tempo médio (min)")
axes[0].set_ylabel("Frequência")
axes[0].legend()
axes[0].grid(True)

axes[1].hist(medias_C, bins=20, edgecolor='black', color='orange')
axes[1].axvline(np.mean(medias_C), color='red', linestyle='--', label=f"Média = {np.mean(medias_C):.1f}")
axes[1].set_title("Distribuição da espera média na etapa C")
axes[1].set_xlabel("Espera média (min)")
axes[1].set_ylabel("Frequência")
axes[1].legend()
axes[1].grid(True)

plt.tight_layout()
plt.show()

---

# Parte 4 — Análise de Sensibilidade: Impacto da Capacidade

Uma pergunta prática: **vale a pena contratar mais um colaborador para a etapa C?**

Vamos comparar duas configurações:
- Config. A: 1 colaborador em C
- Config. B: 2 colaboradores em C

In [ ]:
def simular_salao_config(cap_C, N_rep=50, T=6000):
    """Roda N_rep replicações com capacidade cap_C na etapa C."""
    resultados = []
    for seed in range(N_rep):
        np.random.seed(seed)
        stats = {k: [] for k in ["espera_A", "espera_B", "espera_C",
                                   "proc_A", "proc_B", "proc_C", "tempo_total"]}
        env = simpy.Environment()
        A = simpy.Resource(env, capacity=1)
        B = simpy.Resource(env, capacity=1)
        C = simpy.Resource(env, capacity=cap_C)   # varia aqui
        env.process(chegadas_salao(env, A, B, C, stats))
        env.run(until=T)
        if stats["tempo_total"]:
            resultados.append(np.mean(stats["tempo_total"]))
    return resultados

res_1C = simular_salao_config(cap_C=1)
res_2C = simular_salao_config(cap_C=2)

print(f"Config A (1 em C)  — tempo médio : {np.mean(res_1C):.2f} ± {np.std(res_1C):.2f} min")
print(f"Config B (2 em C)  — tempo médio : {np.mean(res_2C):.2f} ± {np.std(res_2C):.2f} min")
print(f"Redução no tempo total           : {np.mean(res_1C) - np.mean(res_2C):.2f} min")

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
ax.boxplot([res_1C, res_2C], labels=["1 colaborador em C", "2 colaboradores em C"], patch_artist=True,
           boxprops=dict(facecolor="steelblue", alpha=0.6))
ax.set_ylabel("Tempo médio no salão (min)")
ax.set_title("Impacto de adicionar um colaborador na etapa C")
ax.grid(True, axis='y')
plt.tight_layout()
plt.show()

---

# Exercícios

## Exercício 1 — Identificando o Gargalo

No salão de beleza modelado, a etapa B tem processamento fixo de 8 minutos — maior que A (3 min) e similar ao valor esperado de C (4 min).

1. Execute a simulação com diferentes capacidades para B (1, 2, 3 colaboradores) e compare os resultados.
2. Qual configuração reduz mais o tempo total no salão: dobrar a capacidade de B ou dobrar a de C?
3. Interprete o resultado em termos de **identificação de gargalo**.

In [ ]:
def simular_salao_config_B(cap_B, N_rep=50, T=6000):
    resultados = []
    for seed in range(N_rep):
        np.random.seed(seed)
        stats = {k: [] for k in ["espera_A", "espera_B", "espera_C",
                                   "proc_A", "proc_B", "proc_C", "tempo_total"]}
        env = simpy.Environment()
        A = simpy.Resource(env, capacity=1)
        B = simpy.Resource(env, capacity=cap_B)
        C = simpy.Resource(env, capacity=1)
        env.process(chegadas_salao(env, A, B, C, stats))
        env.run(until=T)
        if stats["tempo_total"]:
            resultados.append(np.mean(stats["tempo_total"]))
    return resultados

# Complete abaixo:
res_1B = simular_salao_config_B(cap_B=1)
res_2B = simular_salao_config_B(cap_B=___)

print(f"1 colaborador em B → tempo médio: {np.mean(res_1B):.2f} min")
print(f"2 colaboradores em B → tempo médio: {np.mean(res_2B):.2f} min")

## Exercício 2 — Processo M/M/1 com Múltiplas Replicações

Considere um hospital com:
- $\lambda = 2$ pacientes/hora
- $\mu = 3$ pacientes/hora (1 médico)
- Fórmula analítica: $W_q = \lambda / [\mu(\mu - \lambda)] = 2/3$ hora

1. Rode **50 replicações** de 10.000 horas cada.
2. Plote o histograma do $W_q$ médio por replicação.
3. Calcule o intervalo de confiança de 95% para $W_q$.
4. O IC contém o valor teórico $2/3$?

In [ ]:
def simular_hospital(lam, mu, s, T_sim, seed):
    np.random.seed(seed)
    esperas = []

    def paciente(env, medico):
        chegou = env.now
        with medico.request() as req:
            yield req
            esperas.append(env.now - chegou)
            yield env.timeout(np.random.exponential(1 / mu))

    def chegadas_h(env, medico):
        i = 1
        while True:
            yield env.timeout(np.random.exponential(1 / lam))
            env.process(paciente(env, medico))
            i += 1

    env = simpy.Environment()
    med = simpy.Resource(env, capacity=s)
    env.process(chegadas_h(env, med))
    env.run(until=T_sim)
    return np.mean(esperas) if esperas else 0

# Complete os parâmetros:
wq_reps = [simular_hospital(lam=___, mu=___, s=1, T_sim=10_000, seed=i) for i in range(50)]

media = np.mean(wq_reps)
ep    = 1.96 * np.std(wq_reps) / np.sqrt(len(wq_reps))

print(f"Wq médio           : {media:.4f} h")
print(f"IC 95%             : [{media - ep:.4f}, {media + ep:.4f}]")
print(f"Wq teórico (M/M/1) : {2/(3*(3-2)):.4f} h")

plt.figure(figsize=(8, 4))
plt.hist(wq_reps, bins=15, edgecolor='black', color='steelblue')
plt.axvline(media, color='red', linestyle='--', label=f"Média = {media:.3f}")
plt.axvline(2/3, color='green', linestyle=':', label="Teórico = 0.667")
plt.xlabel("Wq médio por replicação (h)")
plt.ylabel("Frequência")
plt.title("Distribuição de Wq em 50 replicações — M/M/1")
plt.legend()
plt.grid(True)
plt.show()

## Exercício 3 — Projeto Livre

Modele um processo de sua escolha com **pelo menos 3 atividades**, sendo ao menos **uma com roteamento condicional**.

Sugestões:
- Pronto-socorro: triagem → consulta → (exame em 30% dos casos) → alta
- Fábrica: corte → montagem → (inspeção de qualidade, retrabalho em 20%) → embalagem
- Banco: senha → caixa → (gerente em 10% dos casos) → saída

**Entregáveis:**
1. Diagrama do processo (pode ser em texto/ASCII).
2. Código SimPy com coleta de estatísticas por etapa.
3. Gráfico comparando espera e processamento por etapa.
4. Discussão: onde está o gargalo? O que você mudaria para melhorar o desempenho?

In [ ]:
# Trabalhe aqui
